# Multimodal Product Retrieval on ABO
text<->image retrieval, CLIP zero-shot vs LoRA fine-tuned

## setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install awscli boto3 peft --quiet
!pip install -U torchao --quiet

import os
BASE = '/content/drive/MyDrive/Image-Text Retrieval'
os.makedirs(f'{BASE}/listings', exist_ok=True)
os.makedirs(f'{BASE}/images_meta', exist_ok=True)
os.makedirs(f'{BASE}/small_images', exist_ok=True)
os.makedirs(f'{BASE}/embeddings', exist_ok=True)

In [ ]:
import subprocess

def download_if_missing(s3_path, local_path):
    if os.path.exists(local_path):
        return
    result = subprocess.run(['aws', 's3', 'cp', '--no-sign-request', s3_path, local_path],
                             capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {s3_path}")
        print(result.stderr)

## pulling and merging the listing shards

In [ ]:
import gzip, json
import pandas as pd

all_listings = [f'listings_{c}.json.gz' for c in list('0123456789') + list('abcdef')]

def listings_gz_extract(file_name):
    data = []
    with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

def get_combined_files(all_files):
    all_file_df = pd.DataFrame()
    for file_name in all_files:
        download_if_missing(f's3://amazon-berkeley-objects/listings/metadata/{file_name}', f'{BASE}/listings/{file_name}')
        file_df = listings_gz_extract(file_name)
        all_file_df = pd.concat([all_file_df, file_df], ignore_index=True)
    return all_file_df

combined_listing_df = get_combined_files(all_listings)
combined_listing_df.shape

columns are language-tagged (list of {language_tag, value}), value-only, or plain scalar. english marketplaces make up ~79% of the data so filtering to those keeps the text encoder in-distribution

In [ ]:
english_tags = {'en_IN', 'en_US', 'en_CA', 'en_GB', 'en_AU', 'en_AE', 'en_SG'}

def extract_lang_field(raw_row, key, language_codes, join_multi=False):
    key_value = raw_row.get(key)
    if not isinstance(key_value, list):
        return ''
    matches = [pair['value'] for pair in key_value if pair.get('language_tag') in language_codes]
    if not matches:
        return ''
    if join_multi:
        matches = list(dict.fromkeys(matches))
    return ' | '.join(matches) if join_multi else matches[0]

def extract_value_only(raw_row, key):
    key_value = raw_row.get(key)
    if not isinstance(key_value, list) or len(key_value) == 0:
        return ''
    return key_value[0].get('value', '')

def extract_raw(raw_row, key):
    value = raw_row.get(key)
    return value if value is not None else ''

language_cols = {
    'brand': False,
    'bullet_point': True,
    'color': False,
    'item_name': False,
    'model_name': False,
    'item_keywords': True,
}
value_only_cols = ['product_type']
raw_cols = ['item_id', 'country', 'marketplace', 'main_image_id']

data = []
for file_name in all_listings:
    with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
        for line in f:
            raw_row = json.loads(line)
            mod_row = {}
            for col, multi in language_cols.items():
                mod_row[col] = extract_lang_field(raw_row, col, english_tags, join_multi=multi)
            for col in value_only_cols:
                mod_row[col] = extract_value_only(raw_row, col)
            for col in raw_cols:
                mod_row[col] = extract_raw(raw_row, col)
            data.append(mod_row)

df_final = pd.DataFrame(data)
df_final.shape

In [ ]:
df_final = df_final[df_final['marketplace'].isin(['Amazon', 'PrimeNow'])]
df_final.shape

## text field + cleanup

In [ ]:
import numpy as np

brands_df = df_final.copy()
brands_df = brands_df.replace('', np.nan)

cols = ['brand', 'bullet_point', 'color', 'item_name', 'model_name', 'item_keywords']
brands_df = brands_df[~brands_df[cols].isna().all(axis=1)]

brands_df['combined_text'] = brands_df['item_name'] + " " + brands_df['bullet_point']
brands_df = brands_df[~brands_df['combined_text'].isna()]
brands_df = brands_df.drop(columns=['item_name', 'bullet_point'])
brands_df['ct_len'] = brands_df['combined_text'].apply(len)

brands_df = brands_df[~brands_df['main_image_id'].isna()]
brands_df.shape

checking whether item_keywords adds anything beyond title+bullets, since it's SEO-stuffed a lot of the time

In [ ]:
import re

def clean_words(text):
    if not isinstance(text, str) or text.strip() == '':
        return set()
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return set(w for w in text.split() if w)

combined_words = brands_df['combined_text'].apply(clean_words)
keyword_words = brands_df['item_keywords'].apply(clean_words)

overlap_ratio = [
    len(kw & cw) / len(kw) if len(kw) > 0 else np.nan
    for kw, cw in zip(keyword_words, combined_words)
]
pd.Series(overlap_ratio).describe()

median overlap ~62%, and keywords would eat into the 77-token limit for not much new info -> dropping item_keywords from combined_text

## dedup
two separate issues: same product across marketplaces (item_id dupes), and different products sharing a reused placeholder photo (main_image_id collisions). both resolved by keeping the row with the richest text, country priority as tiebreak

In [ ]:
country_priority = ['IN', 'US', 'CA', 'GB', 'DE', 'MX', 'ES', 'FR', 'IT', 'JP',
                     'AU', 'NL', 'SG', 'AE', 'SE', 'SA', 'TR', 'UK', 'PL', 'BR']
priority_map = {c: i for i, c in enumerate(country_priority)}
brands_df['country_rank'] = brands_df['country'].map(priority_map).fillna(len(country_priority))

def dedup_by_group(df, group_col):
    dup_mask = df.groupby(group_col)[group_col].transform('count') > 1
    dup_subset = df[dup_mask].sort_values(by=[group_col, 'ct_len', 'country_rank'], ascending=[True, False, True])
    winners = dup_subset.drop_duplicates(subset=group_col, keep='first')
    losing_indices = dup_subset.index.difference(winners.index)
    return df.drop(index=losing_indices)

brands_df = dedup_by_group(brands_df, 'item_id')
brands_df = dedup_by_group(brands_df, 'main_image_id')
brands_df.shape

## category scope
cellular phone cases are ~60% of the data on their own, would dominate every metric. dropping that category entirely and keeping categories with enough listings for a reliable per-category recall test

In [ ]:
brands_df = brands_df[brands_df['product_type'] != 'CELLULAR_PHONE_CASE']

product_counts = brands_df['product_type'].value_counts()
keep_categories = product_counts[product_counts > 300].index
brands_df = brands_df[brands_df['product_type'].isin(keep_categories)]
brands_df.shape

## images

In [ ]:
download_if_missing('s3://amazon-berkeley-objects/images/metadata/images.csv.gz', f'{BASE}/images_meta/images.csv.gz')
image_df = pd.read_csv(f'{BASE}/images_meta/images.csv.gz', compression='gzip')

brands_img = brands_df.merge(image_df, left_on='main_image_id', right_on='image_id')
brands_img = brands_img[brands_img['image_id'].notna()]
brands_img['path_map'] = brands_img['path'].apply(lambda x: f"{BASE}/small_images/{x.replace('/', '_')}")

brands_img.shape

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor, as_completed

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
BUCKET = 'amazon-berkeley-objects'

def download_one(s3_path, local_path):
    if os.path.exists(local_path):
        return local_path, True
    try:
        s3.download_file(BUCKET, f'images/small/{s3_path}', local_path)
        return local_path, True
    except Exception as e:
        print(f"failed: {s3_path} -> {e}")
        return local_path, False

def download_images_parallel(df, max_workers=20):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(download_one, row['path'], row['path_map']) for _, row in df.iterrows()]
        for future in as_completed(futures):
            future.result()

download_images_parallel(brands_img)

In [ ]:
brands_img.to_csv(f'{BASE}/brands_img.csv')
brands_img.shape

## clip setup

In [ ]:
from transformers import AutoModel, AutoProcessor
from PIL import Image
import torch

model = AutoModel.from_pretrained("openai/clip-vit-base-patch32")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
device

## catalog embeddings
checkpointed since a full pass over 25k images takes a while and colab disconnects mid-run

In [ ]:
def _load_checkpoint(checkpoint_path):
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path)
        return ckpt['embeddings'], ckpt['item_ids'], ckpt['next_row']
    return None, [], 0

def _save_checkpoint(checkpoint_path, prev_embeddings, new_embeddings_list, prev_item_ids, new_ids, next_row):
    new_tensor = torch.cat(new_embeddings_list, dim=0)
    combined_embeddings = torch.cat([prev_embeddings, new_tensor], dim=0) if prev_embeddings is not None else new_tensor
    combined_ids = prev_item_ids + new_ids
    torch.save({'embeddings': combined_embeddings, 'item_ids': combined_ids, 'next_row': next_row}, checkpoint_path)
    return combined_embeddings, combined_ids

In [ ]:
def get_image_embeddings(batch_size, df, checkpoint_path):
    saved_embeddings, saved_item_ids, start_batch = _load_checkpoint(checkpoint_path)
    pending_embeddings, pending_ids = [], []

    for i in range(start_batch, df.shape[0], batch_size):
        ubound = min(i + batch_size, df.shape[0])
        data = df[i:ubound]

        valid_images, valid_ids = [], []
        for p, item_id in zip(data['path_map'], data['item_id']):
            try:
                img = Image.open(p).convert('RGB')
            except Exception:
                continue
            valid_images.append(img)
            valid_ids.append(item_id)

        if not valid_images:
            continue

        model_inputs = processor(images=valid_images, return_tensors="pt")
        model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

        with torch.inference_mode():
            batch_output = model.get_image_features(**model_inputs)
        batch_embeddings = batch_output.pooler_output if hasattr(batch_output, 'pooler_output') else batch_output

        pending_embeddings.append(batch_embeddings.cpu())
        pending_ids.extend(valid_ids)

        saved_embeddings, saved_item_ids = _save_checkpoint(checkpoint_path, saved_embeddings, pending_embeddings, saved_item_ids, pending_ids, next_row=ubound)
        pending_embeddings, pending_ids = [], []
        print(f"processed {ubound}/{df.shape[0]}")

    return saved_embeddings, saved_item_ids


def get_text_embeddings(batch_size, df, checkpoint_path):
    saved_embeddings, saved_item_ids, start_batch = _load_checkpoint(checkpoint_path)
    pending_embeddings, pending_ids = [], []

    for i in range(start_batch, df.shape[0], batch_size):
        ubound = min(i + batch_size, df.shape[0])
        data = df[i:ubound]

        valid_text, valid_ids = [], []
        for text_val, item_id in zip(data['combined_text'], data['item_id']):
            if not isinstance(text_val, str) or text_val.strip() == '':
                continue
            valid_text.append(text_val)
            valid_ids.append(item_id)

        if not valid_text:
            continue

        model_inputs = processor(text=valid_text, return_tensors="pt", truncation=True, max_length=77, padding=True)
        model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

        with torch.inference_mode():
            batch_output = model.get_text_features(**model_inputs)
        batch_embeddings = batch_output.pooler_output if hasattr(batch_output, 'pooler_output') else batch_output

        pending_embeddings.append(batch_embeddings.cpu())
        pending_ids.extend(valid_ids)

        saved_embeddings, saved_item_ids = _save_checkpoint(checkpoint_path, saved_embeddings, pending_embeddings, saved_item_ids, pending_ids, next_row=ubound)
        pending_embeddings, pending_ids = [], []
        print(f"processed {ubound}/{df.shape[0]}")

    return saved_embeddings, saved_item_ids

In [ ]:
image_embeds, image_item_ids = get_image_embeddings(512, brands_img, f'{BASE}/embeddings/img_checkpoint.pt')
text_embeds, text_item_ids = get_text_embeddings(512, brands_img, f'{BASE}/embeddings/text_checkpoint.pt')

image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

np.save(f'{BASE}/embeddings/catalog_image_embeds.npy', image_embeds.cpu().numpy())
np.save(f'{BASE}/embeddings/catalog_text_embeds.npy', text_embeds.cpu().numpy())
pd.DataFrame({'item_id': image_item_ids}).to_csv(f'{BASE}/embeddings/catalog_item_ids.csv', index=False)

quick check everything lines up before trusting it

In [ ]:
def verify_embeddings(npy_path, ids_path, expected_df, label=""):
    embeddings = np.load(npy_path)
    item_ids = pd.read_csv(ids_path)['item_id'].tolist()

    print(f"--- {label} ---")
    print(f"shape: {embeddings.shape}, ids: {len(item_ids)}")
    print(f"dupes: {len(item_ids) - len(set(item_ids))}")
    print(f"missing: {len(set(expected_df['item_id']) - set(item_ids))}")
    norms = np.linalg.norm(embeddings, axis=1)
    print(f"norm range: {norms.min():.4f} - {norms.max():.4f}")

verify_embeddings(f'{BASE}/embeddings/catalog_image_embeds.npy', f'{BASE}/embeddings/catalog_item_ids.csv', brands_img, "images")
verify_embeddings(f'{BASE}/embeddings/catalog_text_embeds.npy', f'{BASE}/embeddings/catalog_item_ids.csv', brands_img, "text")

## retrieval
one function for both directions, embeds the query, ranks the catalog by cosine similarity

In [ ]:
def retrieve_top_k(query, catalog_embed, item_ids, top=10, is_image_to_text=False):
    if is_image_to_text:
        valid_images = []
        for p in query:
            try:
                img = Image.open(p).convert('RGB')
            except Exception:
                continue
            valid_images.append(img)

        if not valid_images:
            return pd.DataFrame()

        model_inputs = processor(images=valid_images, return_tensors="pt")
        model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
        with torch.inference_mode():
            output_embeddings = model.get_image_features(**model_inputs)
    else:
        if not query:
            return pd.DataFrame()
        model_inputs = processor(text=query, return_tensors="pt", truncation=True, max_length=77, padding=True)
        model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
        with torch.inference_mode():
            output_embeddings = model.get_text_features(**model_inputs)

    output_embeddings = output_embeddings.pooler_output if hasattr(output_embeddings, 'pooler_output') else output_embeddings
    output_embeddings = output_embeddings.cpu()
    output_embeddings = output_embeddings / output_embeddings.norm(dim=-1, keepdim=True)

    similarity_mat = torch.matmul(catalog_embed, output_embeddings.T)
    top_candidates_idx = torch.topk(similarity_mat, top, dim=0, largest=True, sorted=True).indices

    top_item_ids = pd.DataFrame()
    for column in range(top_candidates_idx.shape[1]):
        idx = top_candidates_idx[:, column].numpy()
        col = item_ids.iloc[idx].reset_index(drop=True)
        col.name = f'query_{column}'
        top_item_ids = pd.concat([top_item_ids, col], axis=1)

    return top_item_ids

In [ ]:
img_tensor = torch.from_numpy(np.load(f'{BASE}/embeddings/catalog_image_embeds.npy'))
catalog_item_ids = pd.read_csv(f'{BASE}/embeddings/catalog_item_ids.csv')['item_id']

retrieve_top_k(['a photo of shoes'], img_tensor, catalog_item_ids, top=5)

## train / test split
fixed and saved once so every model gets evaluated on the exact same held-out items

In [ ]:
from sklearn.model_selection import train_test_split

train_pool_df, test_df = train_test_split(
    brands_img, test_size=0.20, stratify=brands_img['product_type'], random_state=42
)

test_df[['item_id']].to_csv(f'{BASE}/embeddings/test_query_ids.csv', index=False)
train_pool_df[['item_id']].to_csv(f'{BASE}/embeddings/train_pool_ids.csv', index=False)
print(len(train_pool_df), len(test_df))

## evaluation
bidirectional recall@k, overall and per category

In [ ]:
def evaluate_text_to_image_recall(test_df, catalog_embed, catalog_item_ids, k_values=[1, 5, 10], batch_size=32):
    results = {k: {'hits': 0, 'evaluated': 0} for k in k_values}
    per_category_results = {}
    max_k = max(k_values)
    test_df = test_df.reset_index(drop=True)

    for start in range(0, len(test_df), batch_size):
        end = min(start + batch_size, len(test_df))
        batch = test_df.iloc[start:end]
        queries = batch['combined_text'].tolist()

        top_items = retrieve_top_k(queries, catalog_embed, catalog_item_ids, top=max_k, is_image_to_text=False)
        if top_items.empty:
            continue

        for position, (_, row) in enumerate(batch.iterrows()):
            col_name = f'query_{position}'
            if col_name not in top_items.columns:
                continue
            true_item_id = row['item_id']
            category = row['product_type']
            retrieved_ids = top_items[col_name].tolist()

            if category not in per_category_results:
                per_category_results[category] = {k: {'hits': 0, 'evaluated': 0} for k in k_values}

            for k in k_values:
                results[k]['evaluated'] += 1
                per_category_results[category][k]['evaluated'] += 1
                if true_item_id in retrieved_ids[:k]:
                    results[k]['hits'] += 1
                    per_category_results[category][k]['hits'] += 1

    return results, per_category_results


def evaluate_image_to_text_recall(test_df, catalog_embed, catalog_item_ids, k_values=[1, 5, 10], batch_size=32):
    results = {k: {'hits': 0, 'evaluated': 0} for k in k_values}
    per_category_results = {}
    max_k = max(k_values)
    test_df = test_df.reset_index(drop=True)

    for start in range(0, len(test_df), batch_size):
        end = min(start + batch_size, len(test_df))
        batch = test_df.iloc[start:end]
        queries = batch['path_map'].tolist()

        top_items = retrieve_top_k(queries, catalog_embed, catalog_item_ids, top=max_k, is_image_to_text=True)
        if top_items.empty:
            continue

        for position, (_, row) in enumerate(batch.iterrows()):
            col_name = f'query_{position}'
            if col_name not in top_items.columns:
                continue
            true_item_id = row['item_id']
            category = row['product_type']
            retrieved_ids = top_items[col_name].tolist()

            if category not in per_category_results:
                per_category_results[category] = {k: {'hits': 0, 'evaluated': 0} for k in k_values}

            for k in k_values:
                results[k]['evaluated'] += 1
                per_category_results[category][k]['evaluated'] += 1
                if true_item_id in retrieved_ids[:k]:
                    results[k]['hits'] += 1
                    per_category_results[category][k]['hits'] += 1

    return results, per_category_results

### baseline (zero-shot clip)

In [ ]:
test_ids_df = pd.read_csv(f'{BASE}/embeddings/test_query_ids.csv')
test_df = brands_img[brands_img['item_id'].isin(test_ids_df['item_id'])].reset_index(drop=True)

image_embeds_tensor = torch.tensor(np.load(f'{BASE}/embeddings/catalog_image_embeds.npy'))
text_embeds_tensor = torch.tensor(np.load(f'{BASE}/embeddings/catalog_text_embeds.npy'))

t2i_results_base, t2i_per_category_base = evaluate_text_to_image_recall(test_df, image_embeds_tensor, catalog_item_ids)
i2t_results_base, i2t_per_category_base = evaluate_image_to_text_recall(test_df, text_embeds_tensor, catalog_item_ids)

print("--- text to image ---")
for k, r in t2i_results_base.items():
    print(f"Recall@{k}: {r['hits']}/{r['evaluated']} = {r['hits']/r['evaluated']:.4f}")

print("--- image to text ---")
for k, r in i2t_results_base.items():
    print(f"Recall@{k}: {r['hits']}/{r['evaluated']} = {r['hits']/r['evaluated']:.4f}")

## lora fine-tuning
full fine-tuning risks wrecking the pretrained weights on a dataset this small, so only the attention projections + final projection layers get updated

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    modules_to_save=["visual_projection", "text_projection"],
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
train_pool_ids_df = pd.read_csv(f'{BASE}/embeddings/train_pool_ids.csv')
train_pool_df = brands_img[brands_img['item_id'].isin(train_pool_ids_df['item_id'])]

train_df, val_df = train_test_split(
    train_pool_df, test_size=0.15, stratify=train_pool_df['product_type'], random_state=42
)

train_df[['item_id']].to_csv(f'{BASE}/embeddings/train_ids.csv', index=False)
val_df[['item_id']].to_csv(f'{BASE}/embeddings/val_ids.csv', index=False)
print(len(train_df), len(val_df))

images from drive are slow to read repeatedly across epochs, caching them in memory once up front

In [ ]:
image_cache = {}

def load_one(p):
    try:
        return p, Image.open(p).convert('RGB')
    except Exception:
        return p, None

def preload_images_parallel(df, max_workers=16):
    paths = [p for p in df['path_map'] if p not in image_cache]
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(load_one, p) for p in paths]
        for future in as_completed(futures):
            p, img = future.result()
            if img is not None:
                image_cache[p] = img
    print(f"cached {len(image_cache)} images")

preload_images_parallel(train_df)
preload_images_parallel(val_df)

In [ ]:
import torch.nn.functional as F

def train_one_epoch(model, train_df, optimizer, batch_size=32, print_every=20):
    model.train()
    train_df = train_df.sample(frac=1).reset_index(drop=True)
    total_loss = 0
    n_batches = 0

    for start in range(0, len(train_df), batch_size):
        end = min(start + batch_size, len(train_df))
        batch = train_df.iloc[start:end]

        valid_images, valid_texts = [], []
        for p, text in zip(batch['path_map'], batch['combined_text']):
            img = image_cache.get(p)
            if img is None:
                continue
            valid_images.append(img)
            valid_texts.append(text)

        if len(valid_images) < 2:
            continue

        inputs = processor(images=valid_images, text=valid_texts, return_tensors="pt", truncation=True, max_length=77, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        labels = torch.arange(outputs.logits_per_image.shape[0], device=device)
        loss = (F.cross_entropy(outputs.logits_per_image, labels) + F.cross_entropy(outputs.logits_per_text, labels)) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1
        if n_batches % print_every == 0:
            print(f"  batch {n_batches} | loss: {loss.item():.4f}")

    return total_loss / n_batches


def evaluate_loss(model, val_df, batch_size=32):
    model.eval()
    total_loss = 0
    n_batches = 0
    with torch.no_grad():
        for start in range(0, len(val_df), batch_size):
            end = min(start + batch_size, len(val_df))
            batch = val_df.iloc[start:end]

            valid_images, valid_texts = [], []
            for p, text in zip(batch['path_map'], batch['combined_text']):
                img = image_cache.get(p)
                if img is None:
                    continue
                valid_images.append(img)
                valid_texts.append(text)

            if len(valid_images) < 2:
                continue

            inputs = processor(images=valid_images, text=valid_texts, return_tensors="pt", truncation=True, max_length=77, padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = model(**inputs)
            labels = torch.arange(outputs.logits_per_image.shape[0], device=device)
            loss = (F.cross_entropy(outputs.logits_per_image, labels) + F.cross_entropy(outputs.logits_per_text, labels)) / 2

            total_loss += loss.item()
            n_batches += 1

    return total_loss / n_batches

In [ ]:
from torch.optim import AdamW

model = model.to(device)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5)

n_epochs = 15
for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_df, optimizer)
    val_loss = evaluate_loss(model, val_df)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    torch.save(model.state_dict(), f'{BASE}/embeddings/lora_checkpoint_epoch{epoch}.pt')

epoch 14 had the lowest val loss with no sign of overfitting across the run, using that checkpoint

In [ ]:
checkpoint = torch.load(f'{BASE}/embeddings/lora_checkpoint_epoch14.pt', map_location="cpu")
model.load_state_dict(checkpoint)
model = model.to(device)

## re-embedding the catalog with the fine-tuned model

In [ ]:
image_embeds_ft, image_item_ids_ft = get_image_embeddings(512, brands_img, f'{BASE}/embeddings/img_checkpoint_ft.pt')
text_embeds_ft, text_item_ids_ft = get_text_embeddings(512, brands_img, f'{BASE}/embeddings/text_checkpoint_ft.pt')

image_embeds_ft = image_embeds_ft / image_embeds_ft.norm(dim=-1, keepdim=True)
text_embeds_ft = text_embeds_ft / text_embeds_ft.norm(dim=-1, keepdim=True)

np.save(f'{BASE}/embeddings/catalog_image_embeds_ft.npy', image_embeds_ft.cpu().numpy())
np.save(f'{BASE}/embeddings/catalog_text_embeds_ft.npy', text_embeds_ft.cpu().numpy())

## final comparison

In [ ]:
t2i_results_ft, t2i_per_category_ft = evaluate_text_to_image_recall(test_df, image_embeds_ft, catalog_item_ids)
i2t_results_ft, i2t_per_category_ft = evaluate_image_to_text_recall(test_df, text_embeds_ft, catalog_item_ids)

print("text to image")
for k in [1, 5, 10]:
    base = t2i_results_base[k]['hits'] / t2i_results_base[k]['evaluated']
    ft = t2i_results_ft[k]['hits'] / t2i_results_ft[k]['evaluated']
    print(f"R@{k}: {base:.4f} -> {ft:.4f}")

print("image to text")
for k in [1, 5, 10]:
    base = i2t_results_base[k]['hits'] / i2t_results_base[k]['evaluated']
    ft = i2t_results_ft[k]['hits'] / i2t_results_ft[k]['evaluated']
    print(f"R@{k}: {base:.4f} -> {ft:.4f}")

In [ ]:
rows = []
for category in t2i_per_category_base:
    base10 = t2i_per_category_base[category][10]
    ft10 = t2i_per_category_ft[category][10]
    base_rate = base10['hits'] / base10['evaluated'] if base10['evaluated'] else float('nan')
    ft_rate = ft10['hits'] / ft10['evaluated'] if ft10['evaluated'] else float('nan')
    rows.append({'category': category, 'base_r10': base_rate, 'ft_r10': ft_rate, 'delta': ft_rate - base_rate, 'n': base10['evaluated']})

pd.DataFrame(rows).sort_values('delta', ascending=False)